# QuickCart Market Basket Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/11qA3YNfb4F9Rf-24mocs9IDOia9uPLc5?usp=sharing)

**Goal:** Find products that are often purchased together using Apriori and FP-Growth, then explain how QuickCart could use these patterns.

This notebook follows the project specification and uses the supplied transaction dataset.

## 1. Import libraries and load the data

The transaction file has one row for each product in an order. We will reshape it so each row represents one order.

In [1]:
import warnings
# Suppress DeprecationWarnings from environment/Jupyter modules
warnings.filterwarnings("ignore", category=DeprecationWarning)

from pathlib import Path
import time
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

# Robust data path resolution (Google Colab, local notebooks directory, or repo root)
data_candidates = [
    Path("/content/quickcart_basket_transactions.csv"),
    Path("/content/data/raw/quickcart_basket_transactions.csv"),
    Path("../data/raw/quickcart_basket_transactions.csv"),
    Path("data/raw/quickcart_basket_transactions.csv"),
    Path("../data/quickcart_basket_transactions.csv"),
    Path("data/quickcart_basket_transactions.csv"),
    Path("quickcart_basket_transactions.csv"),
]
DATA_PATH = next((p for p in data_candidates if p.exists()), None)
if DATA_PATH is None:
    DATA_PATH = Path("data/raw/quickcart_basket_transactions.csv")

# Output directory resolution
if Path("/content").exists():
    OUTPUT_DIR = Path("/content/output")
elif Path("../outputs").exists():
    OUTPUT_DIR = Path("../outputs")
elif Path("outputs").exists():
    OUTPUT_DIR = Path("outputs")
else:
    OUTPUT_DIR = Path("./output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

transactions = pd.read_csv(DATA_PATH)
print("Using dataset from:", DATA_PATH.resolve())
print("Outputs directory:", OUTPUT_DIR.resolve())
print("Rows and columns:", transactions.shape)
print("Unique orders:", transactions["order_id"].nunique())
print("Unique products:", transactions["sku_name"].nunique())
print("Stores:", transactions["store_id"].nunique())
print("Date range:", transactions["date"].min(), "to", transactions["date"].max())
display(transactions.head())

Using dataset from: C:\Users\Hustle\Downloads\QuickCart_Market_Basket_Analysis_Abhay\QuickCart_Market_Basket_Analysis\data\raw\quickcart_basket_transactions.csv
Outputs directory: C:\Users\Hustle\Downloads\QuickCart_Market_Basket_Analysis_Abhay\QuickCart_Market_Basket_Analysis\outputs
Rows and columns: (26878, 6)
Unique orders: 5000
Unique products: 60
Stores: 12
Date range: 2026-11-01 to 2026-11-30


,order_id,date,store_id,sku_id,sku_name,category
0,ORD-00001,2026-11-05,ST02,SKU025,Rusk 200g,Snacks
1,ORD-00001,2026-11-05,ST02,SKU055,White Bread 400g,Bakery
2,ORD-00001,2026-11-05,ST02,SKU009,Capsicum 500g,Fruits & Vegetables
3,ORD-00001,2026-11-05,ST02,SKU010,Toned Milk 1L,Dairy
4,ORD-00001,2026-11-05,ST02,SKU029,Energy Drink 250ml,Beverages


## 2. Check and prepare the data

Association-rule mining checks whether an item is present in a basket, not how many units were bought. We therefore keep one row per order and product.

In [2]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

print("Missing values by column:")
print(transactions.isna().sum())

before = len(transactions)
transactions = transactions.dropna(subset=["order_id", "sku_name"])
transactions = transactions.drop_duplicates(subset=["order_id", "sku_name"])
print("Rows removed during cleaning:", before - len(transactions))

basket_sizes = transactions.groupby("order_id")["sku_name"].nunique()
print(f"\nBasket size — mean: {basket_sizes.mean():.2f}; median: {basket_sizes.median():.0f}; min: {basket_sizes.min()}; max: {basket_sizes.max()}")

# Top 5 products by support
product_support = transactions.groupby("sku_name")["order_id"].nunique().div(transactions["order_id"].nunique()).sort_values(ascending=False)
print("\nTop 5 products by support:")
display((product_support.head(5) * 100).rename("Support (%)").round(2).to_frame())

Missing values by column:
order_id    0
date        0
store_id    0
sku_id      0
sku_name    0
category    0
dtype: int64
Rows removed during cleaning: 0

Basket size — mean: 5.38; median: 5; min: 1; max: 16

Top 5 products by support:


,Support (%)
sku_name,
White Bread 400g,25.46
Toned Milk 1L,25.40
Potato Chips 90g,21.84
Cola 750ml,21.64
Toor Dal 1kg,20.68


## 3. Convert orders into a basket matrix

Each row is an order and each column is a product. `True` means the product appeared in that order.

In [3]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

baskets = transactions.groupby("order_id")["sku_name"].apply(list).tolist()
encoder = TransactionEncoder()
basket_array = encoder.fit(baskets).transform(baskets)
basket_matrix = pd.DataFrame(basket_array, columns=encoder.columns_)

print("Basket matrix shape:", basket_matrix.shape)
display(basket_matrix.head())

Basket matrix shape: (5000, 60)


,Air Freshener,Apple 1kg,Banana 1kg,Banana Chips 150g,Bar Soap 125g,Basmati Rice 5kg,Besan 1kg,Brown Bread 400g,Butter 100g,Buttermilk 1L,...,Shampoo 340ml,Spinach 250g,Sugar 1kg,Toilet Cleaner 500ml,Tomato 1kg,Toned Milk 1L,Toor Dal 1kg,Toothpaste 150g,Wheat Flour 5kg,White Bread 400g
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,True
1,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
2,False,False,False,False,True,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False


## 4. Find frequent itemsets with Apriori

The assignment specifies a minimum support of **0.02 (2%)**.

In [4]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

MIN_SUPPORT = 0.02

start = time.perf_counter()
itemsets_apriori = apriori(basket_matrix, min_support=MIN_SUPPORT, use_colnames=True)
apriori_time = time.perf_counter() - start

print(f"Apriori runtime: {apriori_time:.4f} seconds")
print("Frequent itemsets:", len(itemsets_apriori))
display(itemsets_apriori.sort_values("support", ascending=False).head(10))
print("\nItemsets by size:")
display(itemsets_apriori["itemsets"].apply(len).value_counts().sort_index().rename_axis("Number of items").to_frame("Count"))

Apriori runtime: 0.0238 seconds
Frequent itemsets: 256


,support,itemsets
59,0.2546,frozenset({White Bread 400g})
55,0.2540,frozenset({Toned Milk 1L})
136,0.2384,"frozenset({White Bread 400g, Toned Milk 1L})"
44,0.2184,frozenset({Potato Chips 90g})
14,0.2164,frozenset({Cola 750ml})
56,0.2068,frozenset({Toor Dal 1kg})
15,0.1996,frozenset({Cooking Oil 1L})
5,0.1982,frozenset({Basmati Rice 5kg})
89,0.1974,"frozenset({Potato Chips 90g, Cola 750ml})"
13,0.1776,frozenset({Choco Cookies 150g})



Itemsets by size:


,Count
Number of items,
1,60
2,78
3,86
4,30
5,2


## 5. Find frequent itemsets with FP-Growth

FP-Growth uses the same support threshold, so its results can be compared fairly with Apriori.

In [5]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

start = time.perf_counter()
itemsets_fpgrowth = fpgrowth(basket_matrix, min_support=MIN_SUPPORT, use_colnames=True)
fpgrowth_time = time.perf_counter() - start

print(f"FP-Growth runtime: {fpgrowth_time:.4f} seconds")
print("Frequent itemsets:", len(itemsets_fpgrowth))
display(itemsets_fpgrowth.sort_values("support", ascending=False).head(10))

FP-Growth runtime: 0.1182 seconds


Frequent itemsets: 256


,support,itemsets
0,0.2546,frozenset({White Bread 400g})
1,0.2540,frozenset({Toned Milk 1L})
60,0.2384,"frozenset({White Bread 400g, Toned Milk 1L})"
5,0.2184,frozenset({Potato Chips 90g})
6,0.2164,frozenset({Cola 750ml})
7,0.2068,frozenset({Toor Dal 1kg})
8,0.1996,frozenset({Cooking Oil 1L})
9,0.1982,frozenset({Basmati Rice 5kg})
64,0.1974,"frozenset({Potato Chips 90g, Cola 750ml})"
16,0.1776,frozenset({Choco Cookies 150g})


## 6. Compare the algorithms

We compare the itemsets themselves and their support values, as well as the runtime from this notebook run.

In [6]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

ap_support = dict(zip(itemsets_apriori["itemsets"], itemsets_apriori["support"]))
fp_support = dict(zip(itemsets_fpgrowth["itemsets"], itemsets_fpgrowth["support"]))

ap_sets = set(ap_support)
fp_sets = set(fp_support)
common_sets = ap_sets & fp_sets
support_differences = [item for item in common_sets if abs(ap_support[item] - fp_support[item]) > 1e-9]

comparison = pd.DataFrame({
    "Algorithm": ["Apriori", "FP-Growth"],
    "Frequent itemsets": [len(itemsets_apriori), len(itemsets_fpgrowth)],
    "Runtime (seconds)": [round(apriori_time, 4), round(fpgrowth_time, 4)]
})
display(comparison)
print("Itemsets found by both:", len(common_sets))
print("Only in Apriori:", len(ap_sets - fp_sets))
print("Only in FP-Growth:", len(fp_sets - ap_sets))
print("Itemsets with different support values:", len(support_differences))

if apriori_time < fpgrowth_time:
    print("Apriori was faster in this run.")
elif fpgrowth_time < apriori_time:
    print("FP-Growth was faster in this run.")
else:
    print("Both algorithms had the same measured runtime in this run.")
print("Runtime can change with the computer, dataset, and software environment.")

,Algorithm,Frequent itemsets,Runtime (seconds)
0,Apriori,256,0.0238
1,FP-Growth,256,0.1182


Itemsets found by both: 256
Only in Apriori: 0
Only in FP-Growth: 0
Itemsets with different support values: 0
Apriori was faster in this run.
Runtime can change with the computer, dataset, and software environment.


## 7. Generate association rules

A rule has the form **A → B**: when A appears in a basket, B may also appear. The assignment asks for rules with lift at least 1.0.

In [7]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

rules = association_rules(itemsets_apriori, metric="lift", min_threshold=1.0)
rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)

print("Rules generated:", len(rules))
display(rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10))

Rules generated: 980


,antecedents,consequents,support,confidence,lift
0,"frozenset({Face Wash 100ml, Choco Cookies 150g})","frozenset({Hand Sanitizer 200ml, Instant Coffe...",0.0206,0.751825,26.850886
1,"frozenset({Hand Sanitizer 200ml, Instant Coffe...","frozenset({Face Wash 100ml, Choco Cookies 150g})",0.0206,0.735714,26.850886
2,"frozenset({Hand Sanitizer 200ml, Choco Cookies...","frozenset({Face Wash 100ml, Instant Coffee 100g})",0.0206,0.746377,26.848087
3,"frozenset({Face Wash 100ml, Instant Coffee 100g})","frozenset({Hand Sanitizer 200ml, Choco Cookies...",0.0206,0.741007,26.848087
4,"frozenset({Ghee 500ml, Instant Coffee 100g})","frozenset({Poha 500g, Choco Cookies 150g})",0.0234,0.785235,26.001156
5,"frozenset({Poha 500g, Choco Cookies 150g})","frozenset({Ghee 500ml, Instant Coffee 100g})",0.0234,0.774834,26.001156
6,"frozenset({Poha 500g, Instant Coffee 100g})","frozenset({Ghee 500ml, Choco Cookies 150g})",0.0234,0.780000,25.324675
7,"frozenset({Ghee 500ml, Choco Cookies 150g})","frozenset({Poha 500g, Instant Coffee 100g})",0.0234,0.759740,25.324675
8,"frozenset({Poha 500g, White Bread 400g})","frozenset({Ghee 500ml, Toned Milk 1L})",0.0292,0.793478,21.798853
9,"frozenset({Ghee 500ml, Toned Milk 1L})","frozenset({Poha 500g, White Bread 400g})",0.0292,0.802198,21.798853


### The three main metrics

- **Support:** the fraction of all baskets containing both sides of a rule.
- **Confidence:** how often the consequent appears in baskets that contain the antecedent.
- **Lift:** confidence divided by the overall support of the consequent. Lift above 1 means the items occur together more often than expected if they were independent.

These metrics show association, not proof that one product causes another to be purchased.

## 8. Filter actionable rules

The assignment's suggested filter is **support > 0.05** and **lift > 2**.

In [8]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

actionable_rules = rules[(rules["support"] > 0.05) & (rules["lift"] > 2)].copy()
actionable_rules = actionable_rules.sort_values(["support", "lift"], ascending=False)
print("Actionable rules:", len(actionable_rules))
display(actionable_rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10))

Actionable rules: 42


,antecedents,consequents,support,confidence,lift
616,frozenset({Toned Milk 1L}),frozenset({White Bread 400g}),0.2384,0.938583,3.686499
617,frozenset({White Bread 400g}),frozenset({Toned Milk 1L}),0.2384,0.936371,3.686499
352,frozenset({Potato Chips 90g}),frozenset({Cola 750ml}),0.1974,0.903846,4.176738
353,frozenset({Cola 750ml}),frozenset({Potato Chips 90g}),0.1974,0.912200,4.176738
538,frozenset({Toor Dal 1kg}),frozenset({Cooking Oil 1L}),0.1544,0.746615,3.740557
539,frozenset({Cooking Oil 1L}),frozenset({Toor Dal 1kg}),0.1544,0.773547,3.740557
534,frozenset({Basmati Rice 5kg}),frozenset({Toor Dal 1kg}),0.1534,0.773966,3.742581
535,frozenset({Toor Dal 1kg}),frozenset({Basmati Rice 5kg}),0.1534,0.741779,3.742581
292,frozenset({Choco Cookies 150g}),frozenset({Instant Coffee 100g}),0.1520,0.855856,4.835344
293,frozenset({Instant Coffee 100g}),frozenset({Choco Cookies 150g}),0.1520,0.858757,4.835344


## 9. Check the six planted patterns

The assignment provides expected support, confidence, and lift values for six known pairs. The notebook compares those values with the rules found above.

In [9]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

planted_patterns = [
    ("Breakfast Combo", "Toned Milk 1L", "White Bread 400g", 23.84, 93.86, 3.69),
    ("Thali Staples", "Basmati Rice 5kg", "Toor Dal 1kg", 15.34, 77.40, 3.74),
    ("Snack-Drink Combo", "Potato Chips 90g", "Cola 750ml", 19.74, 90.38, 4.18),
    ("Ghar Breakfast", "Ghee 500ml", "Poha 500g", 14.68, 86.05, 5.07),
    ("Tea-Time", "Instant Coffee 100g", "Choco Cookies 150g", 15.20, 85.88, 4.84),
    ("Personal Care Restock", "Hand Sanitizer 200ml", "Face Wash 100ml", 13.18, 79.30, 5.05),
]

validation_rows = []
for name, antecedent, consequent, expected_support, expected_confidence, expected_lift in planted_patterns:
    match = rules[
        (rules["antecedents"] == frozenset([antecedent])) &
        (rules["consequents"] == frozenset([consequent]))
    ]

    if match.empty:
        validation_rows.append({"Pattern": name, "Rule": f"{antecedent} -> {consequent}", "Expected support (%)": expected_support,
                                "Observed support (%)": None, "Expected confidence (%)": expected_confidence,
                                "Observed confidence (%)": None, "Expected lift": expected_lift,
                                "Observed lift": None, "Status": "FAIL — rule not found"})
        continue

    row = match.iloc[0]
    observed_support = row["support"] * 100
    observed_confidence = row["confidence"] * 100
    observed_lift = row["lift"]
    passed = (abs(observed_support - expected_support) < 0.1 and
              abs(observed_confidence - expected_confidence) < 0.1 and
              abs(observed_lift - expected_lift) < 0.05)

    validation_rows.append({"Pattern": name, "Rule": f"{antecedent} -> {consequent}",
                            "Expected support (%)": expected_support, "Observed support (%)": round(observed_support, 2),
                            "Expected confidence (%)": expected_confidence, "Observed confidence (%)": round(observed_confidence, 2),
                            "Expected lift": expected_lift, "Observed lift": round(observed_lift, 2),
                            "Status": "PASS" if passed else "FAIL — metric mismatch"})

validation = pd.DataFrame(validation_rows)
display(validation)
print("Patterns passed:", (validation["Status"] == "PASS").sum(), "out of", len(validation))

,Pattern,Rule,Expected support (%),Observed support (%),Expected confidence (%),Observed confidence (%),Expected lift,Observed lift,Status
0,Breakfast Combo,Toned Milk 1L -> White Bread 400g,23.84,23.84,93.86,93.86,3.69,3.69,PASS
1,Thali Staples,Basmati Rice 5kg -> Toor Dal 1kg,15.34,15.34,77.40,77.40,3.74,3.74,PASS
2,Snack-Drink Combo,Potato Chips 90g -> Cola 750ml,19.74,19.74,90.38,90.38,4.18,4.18,PASS
3,Ghar Breakfast,Ghee 500ml -> Poha 500g,14.68,14.68,86.05,86.05,5.07,5.07,PASS
4,Tea-Time,Instant Coffee 100g -> Choco Cookies 150g,15.20,15.20,85.88,85.88,4.84,4.84,PASS
5,Personal Care Restock,Hand Sanitizer 200ml -> Face Wash 100ml,13.18,13.18,79.30,79.30,5.05,5.05,PASS


Patterns passed: 6 out of 6


## 10. Check the control pair

The assignment states that **Banana 1kg → Shampoo 340ml** should fall below the 2% minimum support.

In [10]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

item_a = "Banana 1kg"
item_b = "Shampoo 340ml"

co_occurrences = int((basket_matrix[item_a] & basket_matrix[item_b]).sum())
total_baskets = len(basket_matrix)
control_support = co_occurrences / total_baskets

print(f"Baskets containing both products: {co_occurrences} of {total_baskets}")
print(f"Support: {control_support:.4f} ({control_support * 100:.2f}%)")
print(f"Minimum support: {MIN_SUPPORT:.2f} ({MIN_SUPPORT * 100:.2f}%)")
if control_support < MIN_SUPPORT:
    print("PASS: The pair is below the minimum support threshold.")
else:
    print("CHECK: The pair meets the minimum support threshold.")

Baskets containing both products: 11 of 5000
Support: 0.0022 (0.22%)
Minimum support: 0.02 (2.00%)
PASS: The pair is below the minimum support threshold.


## 11. Business recommendations

Use the strongest rules as ideas to test, not as guaranteed ways to increase sales.

In [11]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Keep simple one-product-to-one-product rules for shelf and bundle suggestions
pair_rules = actionable_rules[
    actionable_rules["antecedents"].apply(len).eq(1) &
    actionable_rules["consequents"].apply(len).eq(1)
].copy()

seen_pairs = set()
shelf_suggestions = []
for _, row in pair_rules.sort_values(["support", "lift"], ascending=False).iterrows():
    a = next(iter(row["antecedents"]))
    b = next(iter(row["consequents"]))
    pair_key = tuple(sorted([a, b]))
    if pair_key not in seen_pairs:
        seen_pairs.add(pair_key)
        shelf_suggestions.append((a, b, row["support"], row["confidence"], row["lift"]))
    if len(shelf_suggestions) == 5:
        break

print("Possible shelf-placement or bundle ideas:")
if not shelf_suggestions:
    print("No single-product pairs met the selected actionable-rule thresholds.")
else:
    for a, b, support, confidence, lift in shelf_suggestions:
        print(f"- {a} + {b}: support {support:.1%}, confidence {confidence:.1%}, lift {lift:.2f}")

print("\nPossible app cross-sell rules:")
for _, row in pair_rules.sort_values(["confidence", "support"], ascending=False).head(5).iterrows():
    a = next(iter(row["antecedents"]))
    b = next(iter(row["consequents"]))
    print(f"- If a basket contains {a}, consider recommending {b} (confidence {row['confidence']:.1%}, lift {row['lift']:.2f}).")

print("\nThese are suggestions based on past baskets. A store or app test is needed to measure their real effect.")

Possible shelf-placement or bundle ideas:
- Toned Milk 1L + White Bread 400g: support 23.8%, confidence 93.9%, lift 3.69
- Potato Chips 90g + Cola 750ml: support 19.7%, confidence 90.4%, lift 4.18
- Toor Dal 1kg + Cooking Oil 1L: support 15.4%, confidence 74.7%, lift 3.74
- Basmati Rice 5kg + Toor Dal 1kg: support 15.3%, confidence 77.4%, lift 3.74
- Choco Cookies 150g + Instant Coffee 100g: support 15.2%, confidence 85.6%, lift 4.84

Possible app cross-sell rules:
- If a basket contains Toned Milk 1L, consider recommending White Bread 400g (confidence 93.9%, lift 3.69).
- If a basket contains White Bread 400g, consider recommending Toned Milk 1L (confidence 93.6%, lift 3.69).
- If a basket contains Cola 750ml, consider recommending Potato Chips 90g (confidence 91.2%, lift 4.18).
- If a basket contains Potato Chips 90g, consider recommending Cola 750ml (confidence 90.4%, lift 4.18).
- If a basket contains Poha 500g, consider recommending Ghee 500ml (confidence 86.5%, lift 5.07).

These

## 12. Save the results

The following files are saved in the existing `output` folder.

In [12]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

itemsets_apriori.to_csv(OUTPUT_DIR / "frequent_itemsets_apriori.csv", index=False)
itemsets_fpgrowth.to_csv(OUTPUT_DIR / "frequent_itemsets_fpgrowth.csv", index=False)
rules.to_csv(OUTPUT_DIR / "association_rules.csv", index=False)
actionable_rules.to_csv(OUTPUT_DIR / "actionable_rules.csv", index=False)
validation.to_csv(OUTPUT_DIR / "planted_pattern_validation.csv", index=False)

recommendation_lines = [
    "QUICKCART BUSINESS RECOMMENDATIONS",
    "These are ideas based on historical associations, not guaranteed sales outcomes.",
    "",
    "Shelf placement and bundle ideas:"
]
if shelf_suggestions:
    for a, b, support, confidence, lift in shelf_suggestions:
        recommendation_lines.append(f"- {a} + {b} | support: {support:.1%} | confidence: {confidence:.1%} | lift: {lift:.2f}")
else:
    recommendation_lines.append("- No single-product pairs met the selected actionable-rule thresholds.")
recommendation_lines += ["", "App cross-sell ideas:"]
for _, row in pair_rules.sort_values(["confidence", "support"], ascending=False).head(5).iterrows():
    a = next(iter(row["antecedents"]))
    b = next(iter(row["consequents"]))
    recommendation_lines.append(f"- If a basket contains {a}, consider recommending {b} (confidence: {row['confidence']:.1%}, lift: {row['lift']:.2f}).")

(OUTPUT_DIR / "business_recommendations.txt").write_text("\n".join(recommendation_lines), encoding="utf-8")
print("Saved results to:", OUTPUT_DIR.resolve())
print("Done.")

Saved results to: C:\Users\Hustle\Downloads\QuickCart_Market_Basket_Analysis_Abhay\QuickCart_Market_Basket_Analysis\outputs
Done.


## Conclusion

Apriori and FP-Growth find frequent product combinations in the same basket data. Support shows how common a combination is, confidence shows how often the consequent appears when the antecedent is present, and lift compares that pattern with the consequent's usual frequency. The planted-pattern checks and control pair provide simple checks that the results follow the project specification. Business recommendations should be tested before being treated as effective.